# MSTR Enterprise Value and NAV Calculation

This notebook calculates:
1. Enterprise Value (EV) by summing all market caps (shares × price) from strategy.com
2. Current Bitcoin price from Yahoo Finance
3. Net Asset Value (NAV) including cash position
4. Stock price and premium percentage
5. Two NAV scenarios: with and without convertible debt conversion

In [8]:
# ============================================================================
# DATA — load from pre-fetched pipeline cache (no live network calls here)
# Run `python fetch_mstr_treasury.py` to refresh all data.
# ============================================================================
from fetch_mstr_treasury import load_mstr_enriched

mstr_data = load_mstr_enriched()
CURRENT_BITCOIN_PRICE = mstr_data.get("btc_price", 0)

print(f"\nBitcoin: {mstr_data.get('bitcoin_holdings', 0):,} BTC @ ${CURRENT_BITCOIN_PRICE:,.2f}")
print(f"Cash:    ${mstr_data.get('cash', 0):,.0f}")
print(f"MSTR:    ${mstr_data.get('mstr_price', 0):,.2f}")
for _s in ['strc', 'strd', 'stre', 'strk', 'strf']:
    _sh = mstr_data.get(f'{_s}_shares', 0)
    _px = mstr_data.get(f'{_s}_price', 0)
    if _sh:
        print(f"{_s.upper()}:  {_sh:,} shares @ ${_px:,.2f}")

  ✓ Loaded enriched MSTR data from /Users/kai/Documents/Code/BTC Treasury Valuations/output/mstr_enriched_data.json (fetched: 2026-06-30T23:07:06)

Bitcoin: 847,363 BTC @ $58,578.53
Cash:    $2,550,000,000
MSTR:    $86.93
STRC:  104,894,705 shares @ $84.86
STRD:  14,024,221 shares @ $56.28
STRE:  8,828,025 shares @ $91.13
STRK:  14,020,744 shares @ $58.82
STRF:  12,839,689 shares @ $92.70


In [9]:
# ============================================================================
# CALCULATE MARKET CAPS AND ENTERPRISE VALUE (EV)
# ============================================================================

# Get MSTR shares and price
MSTR_SHARES = mstr_data.get('mstr_shares', 0)
MSTR_PRICE = mstr_data.get('mstr_price', 0)

# Calculate MSTR market cap: shares * price
MSTR_MARKET_CAP = MSTR_SHARES * MSTR_PRICE if (MSTR_SHARES > 0 and MSTR_PRICE > 0) else 0

# Calculate preferred stock market caps: shares * price for each series
preferred_series = ['strc', 'strd', 'stre', 'strk', 'strf']
TOTAL_PREFERRED_MARKET_CAP = 0

print("=" * 70)
print("MARKET CAP CALCULATIONS")
print("=" * 70)
print(f"MSTR Common Stock:")
print(f"  Shares: {MSTR_SHARES:,}")
print(f"  Price: ${MSTR_PRICE:,.2f}")
if MSTR_SHARES > 0 and MSTR_PRICE > 0:
    print(f"  Market Cap = {MSTR_SHARES:,} × ${MSTR_PRICE:,.2f} = ${MSTR_MARKET_CAP:,.0f}")
else:
    print(f"  ⚠ Cannot calculate market cap (missing shares or price)")

print(f"\nPreferred Stock:")
for series in preferred_series:
    shares_key = f'{series}_shares'
    price_key = f'{series}_price'
    
    shares = mstr_data.get(shares_key, 0)
    price = mstr_data.get(price_key, 0)
    
    if shares > 0:
        if price > 0:
            mcap = shares * price
            TOTAL_PREFERRED_MARKET_CAP += mcap
            print(f"  {series.upper()}: {shares:,} shares × ${price:,.2f} = ${mcap:,.0f}")
        else:
            print(f"  {series.upper()}: {shares:,} shares (price not available)")

# Get debt and cash for EV calculation
TOTAL_DEBT = mstr_data.get('total_convertible_debt_principal', 0)
CASH = mstr_data.get('cash', 0)

# Calculate Enterprise Value = Market Cap + Debt - Cash
# EV = Equity Market Cap + Preferred Market Cap + Debt - Cash
EQUITY_MARKET_CAP = MSTR_MARKET_CAP + TOTAL_PREFERRED_MARKET_CAP
ENTERPRISE_VALUE = EQUITY_MARKET_CAP + TOTAL_DEBT - CASH

print(f"\n" + "=" * 70)
print("ENTERPRISE VALUE CALCULATION")
print("=" * 70)
print(f"MSTR Common Stock Market Cap: ${MSTR_MARKET_CAP:,.0f}")
print(f"Total Preferred Stock Market Cap: ${TOTAL_PREFERRED_MARKET_CAP:,.0f}")
print(f"Total Equity Market Cap: ${EQUITY_MARKET_CAP:,.0f}")
print(f"Total Convertible Debt: ${TOTAL_DEBT:,.0f}")
print(f"Cash Reserve: ${CASH:,.0f}")
print(f"\nEnterprise Value (EV) = Equity Market Cap + Debt - Cash")
print(f"Enterprise Value (EV) = ${EQUITY_MARKET_CAP:,.0f} + ${TOTAL_DEBT:,.0f} - ${CASH:,.0f}")
print(f"Enterprise Value (EV): ${ENTERPRISE_VALUE:,.0f}")
print("=" * 70)


MARKET CAP CALCULATIONS
MSTR Common Stock:
  Shares: 375,684,000
  Price: $86.93
  Market Cap = 375,684,000 × $86.93 = $32,658,210,235

Preferred Stock:
  STRC: 104,894,705 shares × $84.86 = $8,901,364,730
  STRD: 14,024,221 shares × $56.28 = $789,283,141
  STRE: 8,828,025 shares × $91.13 = $804,480,262
  STRK: 14,020,744 shares × $58.82 = $824,700,158
  STRF: 12,839,689 shares × $92.70 = $1,190,239,131

ENTERPRISE VALUE CALCULATION
MSTR Common Stock Market Cap: $32,658,210,235
Total Preferred Stock Market Cap: $12,510,067,422
Total Equity Market Cap: $45,168,277,657
Total Convertible Debt: $6,713,750,000
Cash Reserve: $2,550,000,000

Enterprise Value (EV) = Equity Market Cap + Debt - Cash
Enterprise Value (EV) = $45,168,277,657 + $6,713,750,000 - $2,550,000,000
Enterprise Value (EV): $49,332,027,657


In [10]:
# ============================================================================
# ANALYZE CONVERTIBLE DEBT
# ============================================================================

BITCOIN_HOLDINGS = mstr_data.get('bitcoin_holdings', 0)
CASH_RESERVE = mstr_data.get('cash', 0)

# Process convertible debt
convertible_debt = mstr_data.get('convertible_debt', [])
TOTAL_CONVERTIBLE_DEBT_PRINCIPAL = 0
CONVERTIBLE_DEBT_ISSUES = []

print("=" * 70)
print("CONVERTIBLE DEBT ANALYSIS")
print("=" * 70)

if convertible_debt:
    for i, debt in enumerate(convertible_debt, 1):
        # Extract principal - use 'notional' first (actual field name), then fallback to others
        principal = debt.get('notional', debt.get('principal', debt.get('face_value', debt.get('amount', 0))))
        # Extract conversion price - use 'strike_price' first (actual field name), then fallback to others
        conversion_price = debt.get('strike_price', debt.get('strikePrice', debt.get('strike', debt.get('conversion_price', debt.get('conversionPrice', debt.get('conversion_strike', 0))))))
        # Calculate conversion ratio from principal and conversion price
        conversion_ratio = 0
        if principal > 0 and conversion_price > 0:
            conversion_ratio = principal / conversion_price
        
        if principal > 0:
            TOTAL_CONVERTIBLE_DEBT_PRINCIPAL += principal
            
            # Get shares on conversion - use fetched data if available, otherwise calculate
            shares_on_conversion = debt.get('shares_on_conversion', 0)
            if shares_on_conversion == 0 or shares_on_conversion is None:
                # Calculate shares if converted (fallback if not fetched from shares page)
                if conversion_price > 0:
                    shares_on_conversion = principal / conversion_price
                elif conversion_ratio > 0:
                    shares_on_conversion = principal * conversion_ratio
            
            CONVERTIBLE_DEBT_ISSUES.append({
                'principal': principal,
                'conversion_price': conversion_price,
                'conversion_ratio': conversion_ratio,
                'shares_on_conversion': shares_on_conversion,
                'will_convert': conversion_price > 0 and MSTR_PRICE > conversion_price
            })
            
            will_convert_str = "YES" if (conversion_price > 0 and MSTR_PRICE > conversion_price) else "NO"
            print(f"Issue {i}:")
            print(f"  Principal: ${principal:,.0f}")
            print(f"  Conversion Price: ${conversion_price:,.2f}")
            if conversion_price > 0:
                print(f"  Current Stock Price: ${MSTR_PRICE:,.2f}")
                print(f"  Will Convert: {will_convert_str}")
                if MSTR_PRICE > conversion_price:
                    print(f"  Shares if Converted: {shares_on_conversion:,.0f}")
            print()
    
    print(f"Total Convertible Debt Principal: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL:,.0f}")
else:
    print("No convertible debt data found in strategy.com")
    print("Note: You may need to manually add debt data if available")

print("=" * 70)


CONVERTIBLE DEBT ANALYSIS
Issue 1:
  Principal: $1,010,000,000
  Conversion Price: $183.19
  Current Stock Price: $86.93
  Will Convert: NO

Issue 2:
  Principal: $1,500,000,000
  Conversion Price: $672.40
  Current Stock Price: $86.93
  Will Convert: NO

Issue 3:
  Principal: $2,000,000,000
  Conversion Price: $433.43
  Current Stock Price: $86.93
  Will Convert: NO

Issue 4:
  Principal: $800,000,000
  Conversion Price: $149.77
  Current Stock Price: $86.93
  Will Convert: NO

Issue 5:
  Principal: $603,750,000
  Conversion Price: $232.72
  Current Stock Price: $86.93
  Will Convert: NO

Issue 6:
  Principal: $800,000,000
  Conversion Price: $204.33
  Current Stock Price: $86.93
  Will Convert: NO

Total Convertible Debt Principal: $6,713,750,000


In [11]:
# ============================================================================
# CALCULATE NAV - SCENARIO 1: WITHOUT DEBT CONVERSION
# Preferred deducted at current market price (not par).
# ============================================================================

BITCOIN_HOLDINGS = mstr_data.get('bitcoin_holdings', 0)
CASH_RESERVE = mstr_data.get('cash', 0)
BITCOIN_VALUE = BITCOIN_HOLDINGS * CURRENT_BITCOIN_PRICE

# NAV = Bitcoin Value + Cash - Preferred at market - Convertible Debt
NAV_NO_CONVERSION = BITCOIN_VALUE + CASH_RESERVE - TOTAL_PREFERRED_MARKET_CAP - TOTAL_CONVERTIBLE_DEBT_PRINCIPAL

NAV_PER_SHARE_NO_CONVERSION = NAV_NO_CONVERSION / MSTR_SHARES if MSTR_SHARES > 0 else 0

print("=" * 70)
print("NAV CALCULATION - SCENARIO 1: WITHOUT DEBT CONVERSION")
print("=" * 70)
print(f"Bitcoin Holdings: {BITCOIN_HOLDINGS:,} BTC")
print(f"Bitcoin Price: ${CURRENT_BITCOIN_PRICE:,.2f}")
print(f"Bitcoin Value: ${BITCOIN_VALUE:,.0f}")
print(f"Cash Reserve: ${CASH_RESERVE:,.0f}")
print(f"Total Preferred Market Value: ${TOTAL_PREFERRED_MARKET_CAP:,.0f}")
print(f"Total Convertible Debt Principal: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL:,.0f}")
print(f"\nNAV = BTC Value + Cash - Preferred Market Value - Convertible Debt")
print(f"NAV = ${BITCOIN_VALUE:,.0f} + ${CASH_RESERVE:,.0f} - ${TOTAL_PREFERRED_MARKET_CAP:,.0f} - ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL:,.0f}")
print(f"NAV = ${NAV_NO_CONVERSION:,.0f}")
print(f"\nMSTR Shares Outstanding: {MSTR_SHARES:,}")
print(f"NAV per Share: ${NAV_PER_SHARE_NO_CONVERSION:,.2f}")
print("=" * 70)

NAV CALCULATION - SCENARIO 1: WITHOUT DEBT CONVERSION
Bitcoin Holdings: 847,363 BTC
Bitcoin Price: $58,578.53
Bitcoin Value: $49,637,279,976
Cash Reserve: $2,550,000,000
Total Preferred Market Value: $12,510,067,422
Total Convertible Debt Principal: $6,713,750,000

NAV = BTC Value + Cash - Preferred Market Value - Convertible Debt
NAV = $49,637,279,976 + $2,550,000,000 - $12,510,067,422 - $6,713,750,000
NAV = $32,963,462,553

MSTR Shares Outstanding: 375,684,000
NAV per Share: $87.74


In [12]:
# ============================================================================
# CALCULATE NAV - SCENARIO 2: WITH DEBT CONVERSION (ONLY IN-THE-MONEY)
# ============================================================================

TOTAL_SHARES_FROM_CONVERSION = 0
TOTAL_DEBT_CONVERTED = 0

for debt in CONVERTIBLE_DEBT_ISSUES:
    if debt['will_convert']:
        TOTAL_SHARES_FROM_CONVERSION += debt['shares_on_conversion']
        TOTAL_DEBT_CONVERTED += debt['principal']

MSTR_SHARES_AFTER_CONVERSION = MSTR_SHARES + TOTAL_SHARES_FROM_CONVERSION

# Converted debt becomes equity — remove it from liabilities; shares expand denominator
NAV_WITH_CONVERSION = BITCOIN_VALUE + CASH_RESERVE - TOTAL_PREFERRED_MARKET_CAP - (TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED)

NAV_PER_SHARE_WITH_CONVERSION = NAV_WITH_CONVERSION / MSTR_SHARES_AFTER_CONVERSION if MSTR_SHARES_AFTER_CONVERSION > 0 else 0

print("=" * 70)
print("NAV CALCULATION - SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)")
print("=" * 70)
print(f"Current MSTR Shares: {MSTR_SHARES:,}")
print(f"Shares from Conversion: {TOTAL_SHARES_FROM_CONVERSION:,.0f}")
print(f"Total Shares After Conversion: {MSTR_SHARES_AFTER_CONVERSION:,.0f}")
print(f"\nDebt Converted: ${TOTAL_DEBT_CONVERTED:,.0f}")
print(f"Debt Remaining: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED:,.0f}")
print(f"\nBitcoin Value: ${BITCOIN_VALUE:,.0f}")
print(f"Cash Reserve: ${CASH_RESERVE:,.0f}")
print(f"Total Preferred Market Value: ${TOTAL_PREFERRED_MARKET_CAP:,.0f}")
print(f"Remaining Convertible Debt: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED:,.0f}")
print(f"\nNAV = BTC Value + Cash - Preferred Market Value - Remaining Debt")
print(f"NAV = ${BITCOIN_VALUE:,.0f} + ${CASH_RESERVE:,.0f} - ${TOTAL_PREFERRED_MARKET_CAP:,.0f} - ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED:,.0f}")
print(f"NAV = ${NAV_WITH_CONVERSION:,.0f}")
print(f"\nMSTR Shares After Conversion: {MSTR_SHARES_AFTER_CONVERSION:,}")
print(f"NAV per Share: ${NAV_PER_SHARE_WITH_CONVERSION:,.2f}")
print("=" * 70)

NAV CALCULATION - SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)
Current MSTR Shares: 375,684,000
Shares from Conversion: 0
Total Shares After Conversion: 375,684,000

Debt Converted: $0
Debt Remaining: $6,713,750,000

Bitcoin Value: $49,637,279,976
Cash Reserve: $2,550,000,000
Total Preferred Market Value: $12,510,067,422
Remaining Convertible Debt: $6,713,750,000

NAV = BTC Value + Cash - Preferred Market Value - Remaining Debt
NAV = $49,637,279,976 + $2,550,000,000 - $12,510,067,422 - $6,713,750,000
NAV = $32,963,462,553

MSTR Shares After Conversion: 375,684,000
NAV per Share: $87.74


In [13]:
# ============================================================================
# CALCULATE STOCK PRICE AND PREMIUM (BOTH SCENARIOS)
# ============================================================================

STOCK_PRICE = MSTR_PRICE

PREMIUM_NO_CONVERSION = ((STOCK_PRICE - NAV_PER_SHARE_NO_CONVERSION) / NAV_PER_SHARE_NO_CONVERSION) * 100 if NAV_PER_SHARE_NO_CONVERSION > 0 else 0
PREMIUM_WITH_CONVERSION = ((STOCK_PRICE - NAV_PER_SHARE_WITH_CONVERSION) / NAV_PER_SHARE_WITH_CONVERSION) * 100 if NAV_PER_SHARE_WITH_CONVERSION > 0 else 0

print("=" * 70)
print("STOCK PRICE AND PREMIUM CALCULATION")
print("=" * 70)
print(f"MSTR Stock Price: ${STOCK_PRICE:,.2f}")
print(f"\nSCENARIO 1: WITHOUT DEBT CONVERSION")
print(f"  NAV per Share: ${NAV_PER_SHARE_NO_CONVERSION:,.2f}")
print(f"  Premium: {PREMIUM_NO_CONVERSION:+.2f}%")
print(f"\nSCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)")
print(f"  NAV per Share: ${NAV_PER_SHARE_WITH_CONVERSION:,.2f}")
print(f"  Premium: {PREMIUM_WITH_CONVERSION:+.2f}%")
print("=" * 70)

STOCK PRICE AND PREMIUM CALCULATION
MSTR Stock Price: $86.93

SCENARIO 1: WITHOUT DEBT CONVERSION
  NAV per Share: $87.74
  Premium: -0.93%

SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)
  NAV per Share: $87.74
  Premium: -0.93%


In [14]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("MSTR VALUATION SUMMARY")
print("=" * 70)

print(f"\nEnterprise Value (EV): ${ENTERPRISE_VALUE:,.0f}")
print(f"  - MSTR Common Stock Market Cap: ${MSTR_MARKET_CAP:,.0f} ({MSTR_SHARES:,} shares × ${MSTR_PRICE:,.2f})")
print(f"  - Preferred Stock Market Value: ${TOTAL_PREFERRED_MARKET_CAP:,.0f}")
print(f"  - Total Equity Market Cap: ${EQUITY_MARKET_CAP:,.0f}")
print(f"  - Total Convertible Debt: ${TOTAL_DEBT:,.0f}")
print(f"  - Cash Reserve: ${CASH:,.0f}")
print(f"  - EV = Equity Market Cap + Debt - Cash = ${EQUITY_MARKET_CAP:,.0f} + ${TOTAL_DEBT:,.0f} - ${CASH:,.0f}")

print(f"\n" + "=" * 70)
print("SCENARIO 1: WITHOUT DEBT CONVERSION")
print("=" * 70)
print(f"Net Asset Value (NAV): ${NAV_NO_CONVERSION:,.0f}")
print(f"  - Bitcoin Value: ${BITCOIN_VALUE:,.0f} ({BITCOIN_HOLDINGS:,} BTC @ ${CURRENT_BITCOIN_PRICE:,.2f})")
print(f"  - Cash Reserve: ${CASH_RESERVE:,.0f}")
print(f"  - Preferred Market Value: ${TOTAL_PREFERRED_MARKET_CAP:,.0f}")
print(f"  - Convertible Debt: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL:,.0f}")
print(f"\nMSTR Shares Outstanding: {MSTR_SHARES:,}")
print(f"NAV per Share: ${NAV_PER_SHARE_NO_CONVERSION:,.2f}")
print(f"Stock Price: ${STOCK_PRICE:,.2f}")
print(f"Premium: {PREMIUM_NO_CONVERSION:+.2f}%")

print(f"\n" + "=" * 70)
print("SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)")
print("=" * 70)
print(f"Net Asset Value (NAV): ${NAV_WITH_CONVERSION:,.0f}")
print(f"  - Bitcoin Value: ${BITCOIN_VALUE:,.0f} ({BITCOIN_HOLDINGS:,} BTC @ ${CURRENT_BITCOIN_PRICE:,.2f})")
print(f"  - Cash Reserve: ${CASH_RESERVE:,.0f}")
print(f"  - Preferred Market Value: ${TOTAL_PREFERRED_MARKET_CAP:,.0f}")
print(f"  - Remaining Convertible Debt: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED:,.0f}")
print(f"  - Debt Converted to Equity: ${TOTAL_DEBT_CONVERTED:,.0f} ({TOTAL_SHARES_FROM_CONVERSION:,.0f} shares)")
print(f"\nMSTR Shares After Conversion: {MSTR_SHARES_AFTER_CONVERSION:,.0f}")
print(f"NAV per Share: ${NAV_PER_SHARE_WITH_CONVERSION:,.2f}")
print(f"Stock Price: ${STOCK_PRICE:,.2f}")
print(f"Premium: {PREMIUM_WITH_CONVERSION:+.2f}%")

print("\n" + "=" * 70)


MSTR VALUATION SUMMARY

Enterprise Value (EV): $49,332,027,657
  - MSTR Common Stock Market Cap: $32,658,210,235 (375,684,000 shares × $86.93)
  - Preferred Stock Market Value: $12,510,067,422
  - Total Equity Market Cap: $45,168,277,657
  - Total Convertible Debt: $6,713,750,000
  - Cash Reserve: $2,550,000,000
  - EV = Equity Market Cap + Debt - Cash = $45,168,277,657 + $6,713,750,000 - $2,550,000,000

SCENARIO 1: WITHOUT DEBT CONVERSION
Net Asset Value (NAV): $32,963,462,553
  - Bitcoin Value: $49,637,279,976 (847,363 BTC @ $58,578.53)
  - Cash Reserve: $2,550,000,000
  - Preferred Market Value: $12,510,067,422
  - Convertible Debt: $6,713,750,000

MSTR Shares Outstanding: 375,684,000
NAV per Share: $87.74
Stock Price: $86.93
Premium: -0.93%

SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)
Net Asset Value (NAV): $32,963,462,553
  - Bitcoin Value: $49,637,279,976 (847,363 BTC @ $58,578.53)
  - Cash Reserve: $2,550,000,000
  - Preferred Market Value: $12,510,067,422
  - Remainin